In [35]:
import os
import re
import gc
import time
import warnings
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
from arch import arch_model
from joblib import Parallel, delayed
from scipy.stats import skew, kurtosis
from sklearn.metrics import (mean_squared_error,r2_score,)
from sklearn.model_selection import (GroupKFold,)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

In [36]:
# =============================================================================
# WINDOW CONFIG
# =============================================================================


INPUT_START = 0
INPUT_END = 480

TARGET_START = 480
TARGET_END = 600

INPUT_LEN = INPUT_END - INPUT_START
TARGET_LEN = TARGET_END - TARGET_START
TOTAL_LEN = max(INPUT_END, TARGET_END)

FORECAST_OFFSET = TARGET_START - INPUT_END
FORECAST_HORIZON = TARGET_END - INPUT_END

WINDOW_TAG = (
    f"in{INPUT_START}_{INPUT_END}"
    f"_target{TARGET_START}_{TARGET_END}"
)

# =============================================================================
# CONFIG
# =============================================================================

INPUT_START = 0
INPUT_END = 480

TARGET_START = 480
TARGET_END = 600

INPUT_LEN = INPUT_END - INPUT_START
TARGET_LEN = TARGET_END - TARGET_START

TOTAL_LEN = max(INPUT_END, TARGET_END)

FORECAST_OFFSET = TARGET_START - INPUT_END
FORECAST_HORIZON = TARGET_END - INPUT_END

EPSILON = 1e-5
RET_CLIP = 0.05
METRIC_EPSILON = 1e-12

SEED = 42

OUTER_FOLDS = 5
INNER_CV_FOLDS = 5

KNN_KS = [3, 5, 10, 20, 40]


SPEC = dict(
    ar=1,
    ma=1,
    p=1,
    q=1,
    dist="normal",
)

ARIMA_MAXITER = 20
GARCH_MAXITER = 20

N_CLUSTERS = 3

N_JOBS = 8

DEV_N_TIME_IDS = None

CACHE_DIR = "cache_nested_cv_cluster_knn"

USE_FEATURE_CACHE = True

PRECOMPUTE_OUTER_FOLD_FEATURES = True

PROJECT_ROOT = Path.cwd()

CLUSTER_CSV_PATH = (
    PROJECT_ROOT.parent
    / "Clustering+Feature engineering"
    / "processed"
    / "clustering"
    / "best_cluster_labels.csv"
)

In [37]:
# =============================================================================
# HELPERS
# =============================================================================

def robust_fill(mat: np.ndarray) -> np.ndarray:
    bad = ~np.isfinite(mat) | (mat <= 0)

    if not bad.any():
        return mat
    out = mat.copy()
    rows = out.shape[0]
    row_idx = np.arange(rows)[:, None]
    fwd = np.where(~bad, row_idx, -1)
    np.maximum.accumulate(fwd, axis=0, out=fwd)
    rev = np.where(~bad, row_idx, rows)
    bwd = rows - 1 - np.maximum.accumulate(
        (rows - 1 - rev)[::-1],
        axis=0,
    )[::-1]

    use = np.where(fwd >= 0, fwd, bwd)
    all_bad = (use < 0) | (use >= rows)
    use = np.where(all_bad, 0, use)
    out = out[use, np.arange(out.shape[1])]
    return np.where(all_bad, 1.0, out)


def _safe_log_return_matrix(mat: np.ndarray) -> np.ndarray:
    safe = np.where(mat > 0, mat, 1e-6).astype(np.float32)
    ret = np.zeros_like(safe)
    ret[1:] = np.log(safe[1:] / safe[:-1])
    np.clip(ret, -RET_CLIP, RET_CLIP, out=ret)
    np.nan_to_num(
        ret,
        copy=False,
        nan=0.0,
        posinf=RET_CLIP,
        neginf=-RET_CLIP,
    )

    return ret


def safe_skew(x: np.ndarray) -> float:
    if np.std(x) <= 1e-12:
        return 0.0
    v = skew(x, bias=False)
    if not np.isfinite(v):
        return 0.0

    return float(v)


def safe_kurt(x: np.ndarray) -> float:
    if np.std(x) <= 1e-12:
        return 0.0
    v = kurtosis(x, bias=False, fisher=True)
    if not np.isfinite(v):
        return 0.0
    
    return float(v)


def safe_autocorr1(x: np.ndarray) -> float:
    if len(x) < 3:
        return 0.0
    a = x[:-1]
    b = x[1:]

    if np.std(a) <= 1e-12:
        return 0.0
    if np.std(b) <= 1e-12:
        return 0.0
    v = np.corrcoef(a, b)[0, 1]
    if not np.isfinite(v):
        return 0.0

    return float(v)

In [38]:
# =============================================================================
# METRICS
# =============================================================================

def metric_pack(y_true, y_pred):
    fp = np.maximum(np.asarray(y_pred).ravel(), METRIC_EPSILON)
    fa = np.maximum(np.asarray(y_true).ravel(), METRIC_EPSILON)

    mse = mean_squared_error(fa, fp)

    err = fp - fa

    pct = err / fa

    ratio = fa / (fp + METRIC_EPSILON)

    try:
        r2 = float(r2_score(fa, fp))
    except Exception:
        r2 = np.nan

    return {
        "mse": float(mse),
        "rmse": float(np.sqrt(mse)),
        "r2": r2,
        "mae": float(np.mean(np.abs(err))),
        "rmspe": float(np.sqrt(np.mean(pct ** 2))),
        "mape": float(np.mean(np.abs(pct))),
        "qlike": float(
            np.mean(
                ratio - np.log(ratio + METRIC_EPSILON) - 1.0
            )
        ),
    }

In [39]:
# =============================================================================
# BASE FEATURES
# =============================================================================

def make_base_features(task):
    r = task["inp_ret"]

    onehot = [0.0] * task["n_clusters"]
    onehot[task["cluster_id"]] = 1.0

    return [
        np.log(task["rv_in"] + EPSILON),
        np.log(task["rv_last120"] + EPSILON),
        np.log(task["rv_scaled"] + EPSILON),

        task["rv_last120"] / (task["rv_in"] + EPSILON),
        task["rv_scaled"] / (task["rv_in"] + EPSILON),

        np.mean(r),
        np.std(r),
        np.mean(np.abs(r)),
        safe_skew(r),
        safe_kurt(r),
        np.max(np.abs(r)),
        safe_autocorr1(r),

        task["cluster_size"] / task["n_stocks"],
    ] + onehot

In [40]:
# =============================================================================
# CLUSTERS
# =============================================================================

def parse_stock_id(col):
    nums = re.findall(r"\d+", str(col))
    return int(nums[-1])


def build_cluster_map(stock_cols):
    cdf = pd.read_csv(CLUSTER_CSV_PATH)

    cdf["stock_id"] = cdf["stock_id"].astype(int)
    cdf["cluster_id"] = cdf["cluster_id"].astype(int)

    stock_to_cluster = dict(
        zip(cdf["stock_id"], cdf["cluster_id"])
    )

    labels = []

    for c in stock_cols:
        sid = parse_stock_id(c)
        labels.append(stock_to_cluster[sid])

    cluster_map = pd.Series(
        labels,
        index=stock_cols,
        dtype=np.int32,
    )

    return cluster_map


In [41]:
# =============================================================================
# ARMA GARCH
# =============================================================================

def fit_armagarch(task):
    inp_ret = np.asarray(task["inp_ret"], dtype=np.float64)

    base = make_base_features(task)

    fallback_rv = task["rv_scaled"]

    fit_features = [
        fallback_rv,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]

    try:
        y = inp_ret * 100.0

        arma_fit = ARIMA(
            y,
            order=(1, 0, 1),
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(
            method_kwargs={"maxiter": ARIMA_MAXITER}
        )

        resid = np.asarray(arma_fit.resid)

        resid = resid[2:]

        resid = resid[np.isfinite(resid)]

        resid = resid - np.mean(resid)

        if len(resid) < 50:
            raise ValueError

        garch = arch_model(
            resid,
            mean="Zero",
            vol="Garch",
            p=1,
            q=1,
            dist="normal",
            rescale=False,
        )

        garch_fit = garch.fit(
            disp="off",
            show_warning=False,
            update_freq=0,
            options={"maxiter": GARCH_MAXITER},
        )

        fc = garch_fit.forecast(
            horizon=FORECAST_HORIZON,
            reindex=False,
        )

        var_fc = np.asarray(
            fc.variance.values.flatten(),
            dtype=np.float64,
        )

        var_fc = np.maximum(var_fc, 0.0)

        target_var_fc = var_fc[
            FORECAST_OFFSET:
            FORECAST_OFFSET + TARGET_LEN
        ]

        garch_forecast_rv = (
            np.sqrt(np.sum(target_var_fc)) / 100.0
        )

        params = garch_fit.params

        fit_features = [
            garch_forecast_rv,

            float(params.get("omega", 0.0)),
            float(params.get("alpha[1]", 0.0)),
            float(params.get("beta[1]", 0.0)),

            float(np.std(resid)),

            float(np.mean(garch_fit.conditional_volatility)),
            float(np.std(garch_fit.conditional_volatility)),
        ]

    except Exception:
        pass

    x = np.asarray(base + fit_features, dtype=np.float32)

    np.nan_to_num(
        x,
        copy=False,
    )

    return {
        "x": x,
        "y_log": task["y_log"],
        "rv_in": task["rv_in"],
        "rv_fut": task["rv_fut"],
        "cluster_id": task["cluster_id"],
        "time_id": task["time_id"],
    }


In [42]:
# =============================================================================
# BUILD TASKS
# =============================================================================

def build_tasks(
    df,
    stock_cols,
    cluster_map,
    time_ids,
):
    cluster_ids = sorted(cluster_map.unique())

    stock_to_pos = {
        s: i
        for i, s in enumerate(stock_cols)
    }

    cluster_stock_idx = {
        cid: np.array(
            [
                stock_to_pos[s]
                for s in cluster_map.index[cluster_map == cid]
            ],
            dtype=np.int32,
        )
        for cid in cluster_ids
    }

    cluster_sizes = {
        cid: len(cluster_stock_idx[cid])
        for cid in cluster_ids
    }

    tasks = []

    df_sub = df[df["time_id"].isin(time_ids)]

    for tid, grp in df_sub.groupby("time_id"):

        mat_raw = grp[stock_cols].values.astype(np.float32)

        if mat_raw.shape[0] < TOTAL_LEN:
            continue

        mat_in = robust_fill(
            mat_raw[INPUT_START:INPUT_END]
        )

        ret_in_mat = _safe_log_return_matrix(mat_in)

        target_context_start = TARGET_START - 1

        mat_target = robust_fill(
            mat_raw[target_context_start:TARGET_END]
        )

        ret_target = _safe_log_return_matrix(mat_target)

        offset = TARGET_START - target_context_start

        ret_fut_mat = ret_target[
            offset:offset + TARGET_LEN
        ]

        for cid in cluster_ids:

            idx = cluster_stock_idx[cid]

            r_in = ret_in_mat[:, idx]
            r_fut = ret_fut_mat[:, idx]

            inp_ret = r_in.mean(axis=1)

            rv_in = (
                np.sqrt(
                    np.einsum("ij,ij->j", r_in, r_in)
                ).mean()
                + EPSILON
            )

            rv_fut = (
                np.sqrt(
                    np.einsum("ij,ij->j", r_fut, r_fut)
                ).mean()
                + EPSILON
            )

            r_last = r_in[-TARGET_LEN:]

            rv_last120 = (
                np.sqrt(
                    np.einsum("ij,ij->j", r_last, r_last)
                ).mean()
                + EPSILON
            )

            rv_scaled = (
                rv_in * np.sqrt(TARGET_LEN / INPUT_LEN)
                + EPSILON
            )

            y_log = np.log(rv_fut / rv_in)

            tasks.append(
                {
                    "time_id": int(tid),
                    "cluster_id": int(cid),

                    "cluster_size": cluster_sizes[cid],
                    "n_clusters": len(cluster_ids),
                    "n_stocks": len(stock_cols),

                    "inp_ret": inp_ret,

                    "rv_in": rv_in,
                    "rv_fut": rv_fut,
                    "rv_last120": rv_last120,
                    "rv_scaled": rv_scaled,

                    "y_log": y_log,
                }
            )

    return tasks

# =============================================================================
# FEATURE BUILD
# =============================================================================

def build_features(
    df,
    stock_cols,
    cluster_map,
    time_ids,
    fold_name,
):
    os.makedirs(CACHE_DIR, exist_ok=True)

    cache_path = (
        Path(CACHE_DIR)
        / f"{fold_name}_features.npz"
    )

    if USE_FEATURE_CACHE and cache_path.exists():

        z = np.load(cache_path)

        return {
            "X": z["X"],
            "y_log": z["y_log"],
            "rv_in": z["rv_in"],
            "rv_fut": z["rv_fut"],
            "cluster_id": z["cluster_id"],
            "time_id": z["time_id"],
        }

    tasks = build_tasks(
        df,
        stock_cols,
        cluster_map,
        time_ids,
    )

    print(f"{fold_name}: tasks={len(tasks):,}")

    rows = Parallel(
        n_jobs=N_JOBS,
        backend="loky",
        batch_size=32,
        verbose=5,
    )(
        delayed(fit_armagarch)(t)
        for t in tasks
    )

    X = np.vstack([r["x"] for r in rows]).astype(np.float32)

    y_log = np.array(
        [r["y_log"] for r in rows],
        dtype=np.float32,
    )

    rv_in = np.array(
        [r["rv_in"] for r in rows],
        dtype=np.float32,
    )

    rv_fut = np.array(
        [r["rv_fut"] for r in rows],
        dtype=np.float32,
    )

    cluster_id = np.array(
        [r["cluster_id"] for r in rows],
        dtype=np.int32,
    )

    time_id = np.array(
        [r["time_id"] for r in rows],
        dtype=np.int32,
    )

    np.savez_compressed(
        cache_path,
        X=X,
        y_log=y_log,
        rv_in=rv_in,
        rv_fut=rv_fut,
        cluster_id=cluster_id,
        time_id=time_id,
    )

    return {
        "X": X,
        "y_log": y_log,
        "rv_in": rv_in,
        "rv_fut": rv_fut,
        "cluster_id": cluster_id,
        "time_id": time_id,
    }

In [43]:
# =============================================================================
# KNN
# =============================================================================

def fit_knn(X_train, y_train, k):
    scaler = StandardScaler()

    Xs = scaler.fit_transform(X_train)

    model = KNeighborsRegressor(
        n_neighbors=min(k, len(X_train)),
        weights="distance",
        metric="euclidean",
        algorithm="brute",
        n_jobs=-1,
    )

    model.fit(Xs, y_train)

    return scaler, model


def predict_knn(scaler, model, X):
    Xs = scaler.transform(X)

    pred = model.predict(Xs)

    return np.clip(pred, -5.0, 5.0)

In [44]:
# =============================================================================
# INNER CV
# =============================================================================

def optimise_k_per_cluster(train_data):
    best_k_by_cluster = {}

    for cid in sorted(np.unique(train_data["cluster_id"])):

        mask = train_data["cluster_id"] == cid

        Xc = train_data["X"][mask]
        yc = train_data["y_log"][mask]

        rvin = train_data["rv_in"][mask]
        rvfut = train_data["rv_fut"][mask]

        groups = train_data["time_id"][mask]

        gkf = GroupKFold(n_splits=INNER_CV_FOLDS)

        best_score = np.inf
        best_k = None

        print(f"\nCluster {cid}")

        for k in KNN_KS:

            scores = []

            for tr_idx, va_idx in gkf.split(Xc, yc, groups):

                scaler, model = fit_knn(
                    Xc[tr_idx],
                    yc[tr_idx],
                    k,
                )

                log_pred = predict_knn(
                    scaler,
                    model,
                    Xc[va_idx],
                )

                rv_pred = (
                    np.exp(log_pred)
                    * rvin[va_idx]
                )

                m = metric_pack(
                    rvfut[va_idx],
                    rv_pred,
                )

                scores.append(m["rmspe"])

            score = np.mean(scores)

            print(
                f"k={k:<4} "
                f"RMSPE={score*100:.4f}%"
            )

            if score < best_score:
                best_score = score
                best_k = k

        best_k_by_cluster[cid] = best_k

        print(
            f"BEST CLUSTER {cid} -> k={best_k}"
        )

    return best_k_by_cluster

# =============================================================================
# TRAIN + TEST
# =============================================================================

def train_and_predict(
    train_data,
    test_data,
    best_k_by_cluster,
):
    preds = np.zeros(
        len(test_data["y_log"]),
        dtype=np.float64,
    )

    for cid in sorted(np.unique(test_data["cluster_id"])):

        train_mask = train_data["cluster_id"] == cid
        test_mask = test_data["cluster_id"] == cid

        scaler, model = fit_knn(
            train_data["X"][train_mask],
            train_data["y_log"][train_mask],
            best_k_by_cluster[cid],
        )

        preds[test_mask] = predict_knn(
            scaler,
            model,
            test_data["X"][test_mask],
        )

    rv_pred = (
        np.exp(preds)
        * test_data["rv_in"]
    )

    return rv_pred

In [45]:
# =============================================================================
# MAIN NESTED CV
# =============================================================================

def run_nested_cv(df):
    stock_cols = [
        c for c in df.columns
        if c not in (
            "time_id",
            "seconds_in_bucket",
        )
    ]

    cluster_map = build_cluster_map(stock_cols)

    print(cluster_map.value_counts())

    time_ids = np.array(
        sorted(df["time_id"].unique())
    )

    if DEV_N_TIME_IDS is not None:
        time_ids = time_ids[:DEV_N_TIME_IDS]

    outer_cv = GroupKFold(
        n_splits=OUTER_FOLDS
    )

    outer_results = []

    dummy_X = np.zeros(len(time_ids))

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(
            dummy_X,
            groups=time_ids,
        ),
        start=1,
    ):
        print("\n" + "=" * 80)
        print(f"OUTER FOLD {fold}")
        print("=" * 80)

        train_ids = time_ids[train_idx]
        test_ids = time_ids[test_idx]

        print(
            f"Train time_ids={len(train_ids)} | "
            f"Test time_ids={len(test_ids)}"
        )

        train_data = build_features(
            df,
            stock_cols,
            cluster_map,
            train_ids,
            fold_name=f"outer{fold}_train",
        )

        test_data = build_features(
            df,
            stock_cols,
            cluster_map,
            test_ids,
            fold_name=f"outer{fold}_test",
        )

        best_k_by_cluster = optimise_k_per_cluster(
            train_data
        )

        print("\nBest k:")
        print(best_k_by_cluster)

        rv_pred = train_and_predict(
            train_data,
            test_data,
            best_k_by_cluster,
        )

        metrics = metric_pack(
            test_data["rv_fut"],
            rv_pred,
        )

        outer_results.append(metrics)

        print("\nOUTER TEST METRICS")

        for k, v in metrics.items():

            if k in ["rmspe", "mape"]:
                print(f"{k:<10}: {v*100:.4f}%")
            else:
                print(f"{k:<10}: {v}")

        gc.collect()

    print("\n" + "=" * 80)
    print("FINAL NESTED CV RESULTS")
    print("=" * 80)

    keys = outer_results[0].keys()

    for k in keys:

        vals = [r[k] for r in outer_results]

        if k in ["rmspe", "mape"]:
            print(
                f"{k:<10}: "
                f"{np.mean(vals)*100:.4f}% "
                f"+/- "
                f"{np.std(vals)*100:.4f}%"
            )
        else:
            print(
                f"{k:<10}: "
                f"{np.mean(vals):.10f} "
                f"+/- "
                f"{np.std(vals):.10f}"
            )

In [46]:
# =============================================================================
# LOAD DATA
# =============================================================================

def load_parquet(path="final.parquet"):
    print(f"Loading {path}")

    df = pd.read_parquet(path)

    print(df.shape)

    return df

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    start = time.perf_counter()

    df = load_parquet("final.parquet")

    run_nested_cv(df)

    elapsed = (
        time.perf_counter() - start
    ) / 60.0

    print(f"\nTOTAL TIME: {elapsed:.2f} min")

Loading final.parquet
(2298000, 114)
1    90
2    12
0    10
Name: count, dtype: int64

OUTER FOLD 1
Train time_ids=3064 | Test time_ids=766

Cluster 0
k=3    RMSPE=12.3559%
k=5    RMSPE=11.7819%
k=10   RMSPE=11.4117%
k=20   RMSPE=11.2127%
k=40   RMSPE=11.1723%
BEST CLUSTER 0 -> k=40

Cluster 1
k=3    RMSPE=7.2677%
k=5    RMSPE=6.9356%
k=10   RMSPE=6.6957%
k=20   RMSPE=6.6070%
k=40   RMSPE=6.6218%
BEST CLUSTER 1 -> k=20

Cluster 2
k=3    RMSPE=12.3846%
k=5    RMSPE=11.6297%
k=10   RMSPE=11.2032%
k=20   RMSPE=11.0194%
k=40   RMSPE=10.9918%
BEST CLUSTER 2 -> k=40

Best k:
{0: 40, 1: 20, 2: 40}

OUTER TEST METRICS
mse       : 2.5222868893865417e-08
rmse      : 0.00015881709257465148
r2        : 0.9761449922633025
mae       : 0.00010246264096104946
rmspe     : 9.6827%
mape      : 7.2717%
qlike     : 0.004719062979854397

OUTER FOLD 2
Train time_ids=3064 | Test time_ids=766

Cluster 0
k=3    RMSPE=12.2323%
k=5    RMSPE=11.7327%
k=10   RMSPE=11.3533%
k=20   RMSPE=11.1206%
k=40   RMSPE=11.110